[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/exercices/seance3_exercices.ipynb)

# Séance 2.3 — Agréger et croiser plusieurs tables

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- filtrer sur plusieurs conditions sans se noyer dans les parenthèses
- classer et extraire un top 5 en une commande
- répondre à « combien par ... ? » avec `groupby`
- calculer plusieurs indicateurs d'un coup avec `agg`
- rassembler trois fichiers en une seule table avec `merge`
- croiser deux dimensions avec un tableau croisé

## Comment ça marche

La feuille compte **deux parties**, à faire dans l'ordre.

**Partie 1 — l'échauffement.** Le code est déjà écrit, il ne reste que les `____` à
remplir. Chaque exercice se termine par une cellule de **vérification** qui vous dit
immédiatement si votre réponse est bonne.

**Partie 2 — les questions.** Une question, une cellule **vide** : à vous d'écrire le
code entier. Il n'y a pas de vérification automatique — on les corrige ensemble en
séance, et la correction est publiée après.

> ⚠️ Si une vérification de la partie 1 affiche `NameError`, c'est que la cellule
au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la,
puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]
print(ventes.shape, clients.shape, produits.shape)

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Filtrer sur deux conditions

> **Votre mission :**
> - Garder les ventes dont le prix dépasse 10 € **et** la quantité dépasse 20.
> - Mettre le nombre de lignes dans `nb_premium`.

In [ ]:
premium = ventes.query("prix > 10 ____ qte > 20")
nb_premium = len(premium)

print(nb_premium)

In [ ]:
verifier("1 - ventes cheres et volumineuses", nb_premium == 38,
         "dans query() on ecrit and, pas &")

### Exercice 2 — Filtrer sur une liste de pays

> **Votre mission :**
> - Compter les clients situés en France, en Allemagne ou en Belgique → `nb_ue`.
> - ⚠️ Guillemets doubles à l'extérieur, simples à l'intérieur.

In [ ]:
nb_ue = len(clients.query("pays ____ ['France', 'Allemagne', 'Belgique']"))

print(nb_ue)

In [ ]:
verifier("2 - clients dans trois pays", nb_ue == 121,
         "le mot-cle est in, et les noms de pays vont entre guillemets simples")

### Exercice 3 — Le meilleur client

> **Votre mission :**
> - Calculer le chiffre d'affaires par client.
> - Mettre l'identifiant du meilleur dans `meilleur_client` et son CA dans `ca_meilleur` (arrondi à 2 décimales).

In [ ]:
ca_client = ventes.groupby("____")["ca"].____()

meilleur_client = ca_client.idxmax()
ca_meilleur = round(ca_client.____(), 2)

print(meilleur_client, ":", ca_meilleur, "euros")

In [ ]:
verifier("3a - meilleur client", meilleur_client == 14911,
         "groupby sur client_id puis sum() sur la colonne ca")
verifier("3b - son chiffre d'affaires", ca_meilleur == 143825.06,
         "idxmax() renvoie l'identifiant, max() renvoie le montant")

### Exercice 4 — Lignes contre commandes

> **Votre mission :**
> - Pour chaque client, calculer le nombre de **lignes** (`nb_lignes`) et le nombre de **commandes distinctes** (`nb_cmd`).
> - Mettre le résultat dans `resume`.
> - Rappel : une commande de 30 articles = 30 lignes, mais 1 commande.

In [ ]:
resume = ventes.groupby("client_id").agg(
    nb_lignes=("cmd_id", "____"),
    nb_cmd=("cmd_id", "____"),
)

print(resume["nb_lignes"].sum(), "lignes |", resume["nb_cmd"].sum(), "commandes")

In [ ]:
verifier("4a - nombre de lignes", resume["nb_lignes"].sum() == 45123,
         "count compte les lignes du groupe")
verifier("4b - nombre de commandes", resume["nb_cmd"].sum() == 1955,
         "nunique compte les valeurs distinctes, count compte les lignes")

### Exercice 5 — La jointure

> **Votre mission :**
> - Joindre `ventes` et `clients` sur `client_id` → `vc`.
> - **Vérifier** que le nombre de lignes n'a pas changé → `nb_apres`.

In [ ]:
vc = ventes.____(clients, on="____")
nb_apres = len(vc)

print(len(ventes), "->", nb_apres)

In [ ]:
verifier("5 - jointure sans perte", nb_apres == 45123,
         "un nombre different signale une cle de jointure non unique")

### Exercice 6 — Le chiffre d'affaires par pays

> **Votre mission :**
> - À partir de `vc`, calculer le CA par pays, trié du plus grand au plus petit → `ca_pays`.
> - Mettre le CA de la France dans `ca_france` (arrondi à 2 décimales).

In [ ]:
ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=____)
ca_france = round(ca_pays["____"], 2)

print(ca_pays.head(3).round(2))
print("France :", ca_france)

In [ ]:
verifier("6 - CA de la France", ca_france == 133984.8,
         "groupby('pays') puis sum() sur ca, et ca_pays['France']")

### Exercice 7 — Le panier moyen par pays

> **Votre mission :**
> - Pour chaque pays : le CA total (`ca`) et le nombre de **commandes distinctes** (`nb_cmd`).
> - Ajouter une colonne `panier` = CA ÷ nombre de commandes, arrondie à 2 décimales.
> - Mettre le panier moyen irlandais dans `panier_irl`.

In [ ]:
parpays = vc.groupby("pays").agg(
    ca=("ca", "sum"),
    nb_cmd=("cmd_id", "____"),
)
parpays["panier"] = (parpays["ca"] / parpays["____"]).round(2)

panier_irl = parpays.loc["Irlande", "panier"]
print(panier_irl)

In [ ]:
verifier("7 - panier moyen irlandais", panier_irl == 1020.33,
         "avec count au lieu de nunique le panier serait ridiculement bas")

### Exercice 8 — Ajouter les produits

> **Votre mission :**
> - Joindre `vc` et `produits` sur `prod_id` → `complet`.
> - Trouver la catégorie qui génère le plus de CA → `cat_top`.

In [ ]:
complet = vc.merge(produits, on="____")
cat_top = complet.groupby("categorie")["ca"].sum().____()

print(len(complet), "lignes | categorie leader :", cat_top)

In [ ]:
verifier("8a - jointure produits", len(complet) == 45123,
         "la cle commune entre vc et produits est prod_id")
verifier("8b - categorie leader", cat_top == "cuisine",
         "groupby('categorie'), sum() sur ca, puis idxmax()")

### Exercice 9 — Le tableau croisé

> **Votre mission :**
> - Croiser `pays` (en lignes) et `segment` (en colonnes), avec la somme du `ca` → `tableau`.
> - Mettre le CA des clients « premium » français dans `fr_premium` (arrondi à 0 décimale).

In [ ]:
tableau = vc.pivot_table(values="ca", index="____", columns="____", aggfunc="sum")

fr_premium = round(tableau.loc["France", "premium"], 0)
print(fr_premium)

In [ ]:
verifier("9 - premium francais", fr_premium == 114433.0,
         "index=pays (lignes), columns=segment (colonnes), aggfunc='sum'")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - Quelle **part du chiffre d'affaires total** l'Irlande représente-t-elle, en % arrondi à 1 décimale ? → `part_irl`
> - Combien de clients irlandais y a-t-il ? → `nb_cli_irl`
> - Regardez les deux chiffres ensemble. Que diriez-vous à un dirigeant ?

In [ ]:
part_irl = round(100 * ca_pays["Irlande"] / ca_pays.____(), 1)
nb_cli_irl = vc.query("pays == 'Irlande'")["client_id"].____()

print(part_irl, "% du CA pour", nb_cli_irl, "clients")

In [ ]:
verifier("10a - part de l'Irlande", part_irl == 22.7,
         "divisez le CA irlandais par ca_pays.sum()")
verifier("10b - clients irlandais", nb_cli_irl == 2,
         "nunique() sur client_id apres avoir filtre sur l'Irlande")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 11 — Deux clés à la fois

> **Votre mission :**
> - Ajouter une colonne `mois` à `complet` (format `2011-10`), puis calculer le CA par **pays et par mois**.
> - Combien de couples pays–mois obtenez-vous ?
> - *Nouveau :* `groupby` accepte une liste — `df.groupby(['pays', 'mois'])`.

### Question 12 — Le produit numéro un de chaque pays

> **Votre mission :**
> - Calculer le CA par pays **et** par produit, puis trier par pays, et à l'intérieur de chaque pays du plus gros CA au plus petit.
> - Afficher la première ligne de chaque pays. Le résultat devrait vous alerter.
> - *Nouveau :* `df.sort_values(['pays', 'ca'], ascending=[True, False])` — deux colonnes, deux sens.

### Question 13 — Combien de pays font 80 % du chiffre d'affaires ?

> **Votre mission :**
> - Calculer la part de chaque pays dans le CA total, en %, triée du plus grand au plus petit.
> - Puis le **cumul** de ces parts, et enfin le nombre de pays nécessaires pour atteindre 80 %.
> - *Nouveau :* `serie.cumsum()` additionne au fur et à mesure.

### Question 14 — Le top 3 de chaque catégorie

> **Votre mission :**
> - Pour **chaque catégorie**, donner les 3 produits qui rapportent le plus.
> - Le principe est celui de la question 12 : on trie d'abord, on prend ensuite les premiers de chaque groupe.

### Question 15 — Le panier moyen, pays par segment

> **Votre mission :**
> - Attention au piège : un panier est une **commande**, pas une ligne. Il faut donc d'abord agréger par `cmd_id`.
> - Construire ensuite un tableau croisé pays × segment contenant le **panier moyen**.
> - Certaines cases sont vides. Est-ce une erreur ?

### Question 16 — La composition du panier, en %

> **Votre mission :**
> - Pour les quatre pays les plus présents, quelle **part** de leurs lignes chaque catégorie représente-t-elle ?
> - Des effectifs bruts ne se comparent pas entre un pays de 20 000 lignes et un pays de 2 000. Des pourcentages, si.
> - *Nouveau :* `pd.crosstab(a, b, normalize='index')` ramène chaque **ligne** à 100 %.

### Question 17 — La jointure qui ment

> **Votre mission :**
> - Ne garder que les produits de la catégorie `cuisine`, puis les joindre à `ventes` de deux façons : un `merge` normal, et un `merge(how='left', indicator=True)`.
> - Combien de lignes chaque version renvoie-t-elle ? Combien la première fait-elle disparaître **sans le dire** ?
> - *Nouveau :* `how='left'` garde toutes les lignes de gauche ; `indicator=True` ajoute une colonne `_merge` qui dit d'où vient chaque ligne.

### Question 18 — Le meilleur panier moyen, à effectif suffisant

> **Votre mission :**
> - Pour chaque client : son CA total, son nombre de commandes distinctes, et son panier moyen.
> - Quel client a le panier moyen le plus élevé ? Puis la même question en ne gardant que ceux d'**au moins 5 commandes**.
> - La réponse change. Laquelle des deux donneriez-vous à un directeur commercial ?

### Question 19 — Le meilleur mois de chaque pays

> **Votre mission :**
> - À partir du CA par pays et par mois de la question 11, construire un tableau avec les pays en lignes et les mois en colonnes.
> - En déduire, pour chaque pays, le **nom du mois** où il réalise son meilleur chiffre d'affaires.
> - *Nouveau :* `serie.unstack()` transforme le second niveau de regroupement en colonnes ; `df.idxmax(axis=1)` cherche le maximum **le long de chaque ligne**.

### Question 20 — Question de synthèse

> **Votre mission :**
> - Le comité veut savoir **sur quels marchés l'entreprise est vraiment exposée**.
> - Produire un tableau par pays avec : le CA, le nombre de clients distincts, et le CA moyen par client.
> - Le trier par CA décroissant et ne garder que les cinq premiers.
> - Puis, en commentaire, la phrase que vous mettriez sous ce tableau.